# Équité territoriale — Subventions aux communes de l'arrondissement de Lille

Comparaison entre ce que chaque commune a reçu et ce qu'elle aurait reçu selon son poids démographique, au sein de l'arrondissement de Lille (595).

In [ ]:
import pandas as pd

# Chargement des 3 sources
territoire = pd.read_csv('../data/liste_territoire.csv')
financement = pd.read_csv('../data/financement.csv')
operation = pd.read_csv('../data/opérations.csv')
population = pd.read_csv('../data/populations_communes_nord_2022.csv', dtype={'COM': str})

## 1. Filtrer les communes bénéficiaires

In [ ]:
# Garder uniquement les communes de l'arrondissement de Lille (595)
communes = territoire[territoire['type_territoire'] == 'COM'].copy()
communes = communes[communes['insee_ardt'] == 595]
communes['COM'] = communes['insee_siege'].dropna().astype(int).astype(str)
print(f"{len(communes)} communes dans l'arrondissement de Lille")

## 2. Préparer les financements actifs avec fund_type

In [ ]:
# Exclure les lignes annulées
financement_actif = financement[financement['is_annule'] != True].copy()

# Classification fund_type
def attribution_fund_type(type_fct):
    t = type_fct.lower()
    if 'friche' in t: return 'RF'
    if 'vert' in t: return 'FV'
    if 'fnadt' in t: return 'FNADT'
    if 'dpv' in t: return 'DPV'
    if 'dsil' in t: return 'DSIL'
    if 'detr' in t: return 'DETR'
    return 'other'

financement_actif['fund_type'] = financement_actif['type_fct'].apply(attribution_fund_type)

# Jointure avec operation pour récupérer sirene
fin_op = financement_actif.merge(operation[['id_operation', 'sirene', 'programmation']], on='id_operation', how='left')

## 3. Ne garder que les subventions aux communes

In [ ]:
# Jointure avec communes (inner = seulement les communes)
fin_communes = fin_op.merge(communes[['sirene', 'libelle_territoire', 'COM']], on='sirene', how='inner')
print(f"{len(fin_communes)} lignes de financement vers des communes")
print(f"Montant total : {fin_communes['montant_sub'].sum():,.0f} €")

## 4. Agrégation par commune

In [ ]:
# Total reçu par commune (toutes années, tous dispositifs)
sub_par_commune = fin_communes.groupby(['COM', 'libelle_territoire'])['montant_sub'].sum().reset_index()
sub_par_commune.columns = ['COM', 'commune', 'montant_recu']
sub_par_commune = sub_par_commune.sort_values('montant_recu', ascending=False)
print(sub_par_commune.head(10))

## 5. Calcul de l'équité

In [ ]:
# Jointure avec population — uniquement les communes de l'arrondissement
analyse = sub_par_commune.merge(population[['COM', 'Commune', 'PMUN']], on='COM', how='left')
analyse['PMUN'] = pd.to_numeric(analyse['PMUN'])

# Totaux intra-arrondissement uniquement
total_sub = analyse['montant_recu'].sum()
total_pop = analyse['PMUN'].sum()

analyse['poids_demo'] = analyse['PMUN'] / total_pop
analyse['montant_theorique'] = analyse['poids_demo'] * total_sub
analyse['ecart'] = analyse['montant_recu'] - analyse['montant_theorique']
analyse['ecart_pct'] = analyse['ecart'] / analyse['montant_theorique'] * 100

analyse = analyse.sort_values('ecart', ascending=False)
print(analyse[['commune', 'PMUN', 'montant_recu', 'montant_theorique', 'ecart']].head(10))

## 6. Visualisation

In [ ]:
import matplotlib.pyplot as plt

analyse_triee = analyse.sort_values('ecart')
n = len(analyse_triee)

fig, ax = plt.subplots(figsize=(12, max(10, n * 0.3)))

couleurs = ['steelblue' if x >= 0 else 'tomato' for x in analyse_triee['ecart']]
ax.barh(analyse_triee['commune'], analyse_triee['ecart'] / 1e6, color=couleurs)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title("Équité territoriale — Arrondissement de Lille\nÉcart entre subventions reçues et quote-part démographique (M€)")
ax.set_xlabel('Écart (M€)')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.show()